# NBA Fantasy Draft Optimiser - Project Overview

A draft assistant for a **Yahoo head-to-head (H2H) points-format** fantasy NBA league
using a **snake draft**. It projects each player's season value, builds a ranked draft
board, simulates the draft, and validates the whole thing on past seasons **without
data leakage**.

> **Honest framing.** Drafting is only a modest lever on H2H success - in-season
> management (waivers, streaming, trades) usually matters more. The value here is a
> demonstration of a multi-source pipeline, a defensible projection, a constrained
> decision engine, a simulation, and a leakage-free backtest. It is **not** a
> guaranteed league-winning edge, and nothing below claims otherwise.

**Contents**
1. [Objective](#1-objective)
2. [Data wrangling and cleaning](#2-data-wrangling-and-cleaning)
3. [Methodologies](#3-methodologies)
4. [Results](#4-results)
5. [Discussion](#5-discussion)
6. [Conclusion](#6-conclusion)

*Code cells are illustrative extracts from `src/`; charts are rendered by
`python src/main.py` into `outputs/`.*

## 1. Objective

**Build a draft assistant that measurably beats the drafts a human would otherwise
make, and prove it on held-out seasons.**

Concretely, four questions the project has to answer with evidence:

| # | Question | How it is answered |
|---|---|---|
| 1 | Can we project a player's season value? | Time-ordered MAE/RMSE against actuals |
| 2 | Does a smarter projection beat a naive one? | Compare each version against the one below it |
| 3 | Does a smarter draft engine beat greedy? | Simulated drafts scored on actual points |
| 4 | Is the edge real or noise? | Multi-season distribution vs named benchmarks |

**Design constraints (self-imposed, non-negotiable):**

- **No leakage.** A projection for season *N* may use only data from seasons before *N*,
  enforced by assertions and tests, not convention.
- **Always benchmark.** No number is reported in isolation; everything is a margin
  against a baseline.
- **Never fabricate.** API column names are verified at runtime, never assumed.
- **Match complexity to need.** Each layer is kept only if it beats the simpler version.

The project was built in five strict versions, each adding exactly one layer:
V1 thin slice -> V2 better projection + scarcity -> V3 smarter engine ->
V4 aging/shrinkage -> V5 live league data, multi-season validation, presentation.

## 2. Data wrangling and cleaning

### Sources

| Source | Endpoint | Grain | Volume | Used for |
|---|---|---|---|---|
| `nba_api` | `PlayerGameLogs` | one row per player-game | ~26k rows/season | production stats |
| `nba_api` | `CommonTeamRoster` | one row per player-season | ~500/season (30 calls) | positions |
| `nba_api` | `LeagueDashPlayerBioStats` | one row per player-season | ~540/season (1 call) | age |
| Yahoo (`yfpy`) | league settings / draft | league-level | - | live scoring, eligibility, ADP |

### Rule: verify the schema, never assume it

The single most important ingestion decision. Rather than hardcoding guessed column
names, one season was pulled first and the **actual** 70-column schema inspected; the
names used by the pipeline are pinned as constants and validated on every load, so a
silent API change fails loudly instead of corrupting results.

In [ ]:
# src/columns.py - names taken from a REAL pull, not assumed
COL_SEASON      = "SEASON_YEAR"     # e.g. "2022-23"
COL_PLAYER_ID   = "PLAYER_ID"
COL_PLAYER_NAME = "PLAYER_NAME"
COL_GAME_ID     = "GAME_ID"
COL_PTS, COL_REB, COL_AST = "PTS", "REB", "AST"
COL_STL, COL_BLK, COL_TOV = "STL", "BLK", "TOV"

REQUIRED_RAW_COLUMNS = frozenset({
    COL_SEASON, COL_PLAYER_ID, COL_PLAYER_NAME, COL_GAME_ID, ...
})

def validate_schema(df, *, season):
    """Fail loudly if the API schema drifts - never guess a column."""
    missing = REQUIRED_RAW_COLUMNS - set(df.columns)
    if missing:
        raise KeyError(f"{season} pull missing {sorted(missing)} - update columns.py")
    return df

### Rule: be polite to the API, and cache everything

One league-wide call per season (not per player), a short sleep between live calls,
retry-with-exponential-backoff, and a Parquet cache so nothing is ever re-pulled.

In [ ]:
def load_player_logs(seasons: list[str]) -> pd.DataFrame:
    frames = []
    for season in seasons:
        cache = DATA_DIR / f"logs_{season}.parquet"
        if cache.exists():                       # never re-pull
            df = validate_schema(pd.read_parquet(cache), season=season)
        else:
            time.sleep(SLEEP_BETWEEN_CALLS)      # rate-limit courtesy
            df = _pull_one_season_live(season)   # retries w/ backoff inside
            df.to_parquet(cache, index=False)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

### Cleaning decisions and the problems that forced them

Five real data problems surfaced during the build, each with a deliberate fix:

| Problem | Why it mattered | Fix |
|---|---|---|
| **Non-ASCII names** (Jokić, Dončić) crashed console output on Windows cp1252 | pipeline died at the final print | force UTF-8 stdout; strip accents for joins |
| **Missing positions** for 208 board players | value-over-replacement needs a position | source positions across the *whole* projection window, not one season; unmatched get an overall baseline and are **counted, never dropped** |
| **Retired players resurrected** into drafts (Aldridge, Cousins - all scored 0.0) | a multi-season window includes players who left the league | treat a fully-missed season as **0 games** in the durability model |
| **Sophomores penalised** by that same zero-fill | they were not absent, they did not exist yet | only zero-fill seasons **after** a player's first appearance |
| **COVID-shortened seasons** (2019-20, 2020-21) | fewer games distort rate and durability estimates | games-played weighting absorbs it; flagged as a caveat |

### Transformation: raw game logs to season totals

Scoring weights are validated against real columns so a typo raises rather than
silently scoring zero.

In [ ]:
def apply_scoring(logs: pd.DataFrame, scoring: dict[str, float]) -> pd.DataFrame:
    _validate_scoring_keys(scoring, logs)        # typo -> KeyError, not silent 0
    weights = pd.Series(scoring, dtype="float64")
    return logs.assign(**{
        FANTASY_PTS: lambda df: df[list(weights.index)].mul(weights, axis=1).sum(axis=1)
    })

def season_totals(logs: pd.DataFrame) -> pd.DataFrame:
    return (
        logs
        .groupby([COL_SEASON, COL_PLAYER_ID, COL_PLAYER_NAME], as_index=False)
        .agg(TOTAL_FANTASY_PTS=(FANTASY_PTS, "sum"),
             GP=(COL_GAME_ID, "nunique"))
        .sort_values("TOTAL_FANTASY_PTS", ascending=False, ignore_index=True)
    )

### The subtlest cleaning step: absence *is* data

A season a player missed entirely carries information - it is a durability signal, not
a gap to ignore. But only after he entered the league. This one grid-and-fill step was
the difference between a sane draft and one that drafted retired players.

In [ ]:
# Build a complete (player x window-season) grid so a MISSED season counts as 0 games,
# but only from each player's first appearance onward (sophomores are not punished
# for seasons before they existed).
grid = (
    players
    .merge(first_season, on=COL_PLAYER_ID)
    .merge(pd.DataFrame({COL_SEASON: window}), how="cross")
    .loc[lambda df: df[COL_SEASON] >= df["_first_season"]]
)
filled = (
    grid.merge(used, on=[COL_PLAYER_ID, COL_SEASON], how="left")
        .fillna({TOTAL_FANTASY_PTS: 0.0, GP: 0})   # absence == 0 games played
)

### Cross-source identity: the `nba_api` <-> Yahoo join

The two sources use different player ids, so names are normalised (accents,
punctuation and generational suffixes stripped). Anything **ambiguous on either side
is surfaced for manual review**, never mis-joined on a guess and never dropped.

In [ ]:
from src.reconcile import normalise_name, build_crosswalk

normalise_name("Nikola Jokić")        # -> 'nikola jokic'
normalise_name("Jaren Jackson Jr.")   # -> 'jaren jackson'
normalise_name("De'Aaron Fox")        # -> 'de aaron fox'

crosswalk = build_crosswalk(nba_players, yahoo_players)
print(crosswalk.summary())
# e.g. '412 matched, 18 nba-only, 6 yahoo-only (review these)'

## 3. Methodologies

Five versions, each adding exactly one layer and each required to prove itself
against the version below it.

| Version | Adds | Kept? |
|---|---|---|
| **V1** | Prior-season total; greedy snake draft | Baseline (still champion on accuracy) |
| **V2** | Recency-weighted rate x projected games; value over replacement | Projection **no**; VOR board **yes** |
| **V3** | Marginal lineup value + positional-run risk | **Yes** |
| **V4** | Aging curve; empirical-Bayes shrinkage | Aging **yes**; shrinkage **no** |
| **V5** | Live Yahoo settings; reconciliation; multi-season distribution; charts | **Yes** |

### 3.0 Notation

Every formula below uses this notation.

| Symbol | Meaning |
|---|---|
| $i$ | a player |
| $s$ | a season (e.g. `2022-23`); $N$ is the **target** season being projected |
| $g$ | a game |
| $c \in \mathcal{C}$ | a box-score stat (PTS, REB, AST, ...) |
| $w_c$ | the league's fantasy point weight for stat $c$ |
| $x_{i,s,g,c}$ | player $i$'s value of stat $c$ in game $g$ of season $s$ |
| $F_{i,s,g}$ | fantasy points scored in that game |
| $G_{i,s}$ | games played |
| $T_{i,s}$ | season total fantasy points |
| $r_{i,s} = T_{i,s}/G_{i,s}$ | per-game rate |
| $\hat{y}_i(N)$ | the projection for player $i$ in season $N$ |
| $T$ | number of teams in the league |
| $p(i) \in \{G, F, C, \text{UNK}\}$ | player $i$'s coarse position |
| $v_{(j)}$ | the $j$-th **largest** value in a set (order statistic) |

### 3.1 Scoring: box score to fantasy points

The league's scoring formula is a linear functional of the box score. This linearity
is the reason an H2H **points** league is tractable at all: expected team score is
additive over players, so a roster's value is a sum, not an interaction.

$$F_{i,s,g} \;=\; \sum_{c \in \mathcal{C}} w_c \, x_{i,s,g,c}$$

$$T_{i,s} \;=\; \sum_{g} F_{i,s,g}, \qquad\qquad G_{i,s} \;=\; \bigl|\{\, g : \text{player } i \text{ appeared} \,\}\bigr|$$

With the league weights used here, $w_{\text{PTS}}=1$, $w_{\text{REB}}=1.25$,
$w_{\text{AST}}=1.5$, $w_{\text{STL}}=2$, $w_{\text{BLK}}=1.5$, $w_{\text{TOV}}=-1$,
plus made/attempted shot terms. $G_{i,s}$ is a **distinct-game count**, so a player
traded mid-season is not double-counted.

In [ ]:
def apply_scoring(logs: pd.DataFrame, scoring: dict[str, float]) -> pd.DataFrame:
    _validate_scoring_keys(scoring, logs)        # typo -> KeyError, not a silent 0
    weights = pd.Series(scoring, dtype="float64")
    return logs.assign(**{
        FANTASY_PTS: lambda df: df[list(weights.index)].mul(weights, axis=1).sum(axis=1)
    })

def season_totals(logs):
    return (logs.groupby([COL_SEASON, COL_PLAYER_ID, COL_PLAYER_NAME], as_index=False)
                .agg(TOTAL_FANTASY_PTS=(FANTASY_PTS, "sum"),
                     GP=(COL_GAME_ID, "nunique")))     # distinct games

### 3.2 The non-negotiable constraint: time-ordering

Every projection is a function of **strictly prior** data only:

$$\hat{y}_i(N) \;=\; f\!\left(\mathcal{D}_{<N}\right), \qquad
\mathcal{D}_{<N} \;=\; \bigl\{\, (i,s) \;:\; s < N \,\bigr\}$$

Train/test splits are therefore always time-ordered, never random across time. This
is enforced with an assertion on the rows **about to be consumed**, so that loosening
a filter trips the guard rather than silently leaking the future.

In [ ]:
def project_v1(totals, target_season):
    prev = prior_season(target_season)
    used = totals.loc[totals[COL_SEASON] == prev]     # rows actually consumed

    # Protective, not tautological: if the selection above were ever loosened
    # (e.g. to `<= target`), target-season rows would slip in and this trips.
    leaked = used.loc[used[COL_SEASON] >= target_season]
    assert leaked.empty, f"Leakage: {len(leaked)} rows from {target_season} or later"

    return used.rename(columns={TOTAL_FANTASY_PTS: PROJECTED_PTS})

Tested two ways: the projection is unchanged when target-season rows are added to the
input, **and** the guard genuinely fires when a regression is forced.

### 3.3 V1 model - prior-season total

The naive baseline, and the one that proved hardest to beat:

$$\boxed{\;\hat{y}^{\,V1}_i(N) \;=\; T_{i,\,N-1}\;}$$

**Why it is strong.** It is a single number that already embeds *both* halves of the
problem - scoring rate **and** availability - because a player who missed 30 games
simply has a smaller total. It implicitly assumes role stationarity
($r_{i,N} \approx r_{i,N-1}$) and durability persistence
($G_{i,N} \approx G_{i,N-1}$).

**Where it fails.** It cannot separate the two: a star who was injured last season is
indistinguishable from a healthy but mediocre one. It also has no signal for rookies.
Everything from V2 on is an attempt to fix precisely this conflation.

### 3.4 V2 model - recency-weighted rate x projected durability

V2 decomposes the total into a **rate** and a **volume**, estimated separately:

$$\hat{y}^{\,V2}_i(N) \;=\; \underbrace{\hat{r}_i}_{\text{scoring rate}} \times \underbrace{\hat{G}_i}_{\text{projected games}}$$

**The window.** Let $\mathcal{W}_N$ be the $n$ most recent seasons before $N$
($n = 3$), ordered most-recent-first, and let $\rho(s) \in \{0,1,\dots,n-1\}$ be a
season's recency rank ($0$ = most recent):

$$\mathcal{W}_N = \{s_1 > s_2 > \dots > s_n\}, \quad s_j < N, \quad \rho(s_j) = j-1$$

**Absence is data (the zero-fill).** A season the player missed *entirely* must count
as $G=0$ - that is the durability signal - but only from his first appearance onward,
so a player who entered the league mid-window is not punished for seasons before he
existed. With $\text{first}_i = \min\{s \in \mathcal{W}_N : i \text{ appears}\}$:

$$\mathcal{S}_i \;=\; \bigl\{\, s \in \mathcal{W}_N \;:\; s \ge \text{first}_i \,\bigr\},
\qquad G_{i,s} = T_{i,s} = 0 \;\text{ if } i \text{ is absent in } s \in \mathcal{S}_i$$

**Rate** - pooled over the window, which makes it automatically games-weighted (a
10-game season contributes a tenth of the weight of a 100-game one):

$$\hat{r}_i \;=\; \frac{\sum_{s \in \mathcal{S}_i} T_{i,s}}{\sum_{s \in \mathcal{S}_i} G_{i,s}}$$

**Projected games** - a recency-weighted mean with geometric decay
$\delta = 0.75$ per year older:

$$\hat{G}_i \;=\; \frac{\sum_{s \in \mathcal{S}_i} \delta^{\,\rho(s)}\, G_{i,s}}
                       {\sum_{s \in \mathcal{S}_i} \delta^{\,\rho(s)}}$$

Note the deliberate asymmetry: the **rate** is games-weighted (reliability), the
**volume** is recency-weighted (durability trends). The zeros enter only through
$\hat{G}_i$, leaving $\hat{r}_i$ an estimate of ability *when on court*.

In [ ]:
# The zero-fill grid - the subtlest step in the whole projection.
grid = (players
        .merge(first_season, on=COL_PLAYER_ID)
        .merge(pd.DataFrame({COL_SEASON: window}), how="cross")
        .loc[lambda df: df[COL_SEASON] >= df["_first_season"]])   # only after debut

filled = (grid.merge(used, on=[COL_PLAYER_ID, COL_SEASON], how="left")
              .fillna({TOTAL_FANTASY_PTS: 0.0, GP: 0}))           # absence == 0 games

filled = filled.assign(_w=lambda d: d[COL_SEASON].map(season_rank).rpow(GP_DECAY),
                       _wgp=lambda d: d["_w"] * d[GP])

out = (filled.groupby([COL_PLAYER_ID, COL_PLAYER_NAME], as_index=False)
       .agg(_pts=(TOTAL_FANTASY_PTS, "sum"), _gp=(GP, "sum"),
            _wsum=("_w", "sum"), _wgp=("_wgp", "sum"))
       .assign(PROJECTED_PPG=lambda d: d._pts / d._gp,            # rate  r_hat
               PROJECTED_GP=lambda d: d._wgp / d._wsum,           # games G_hat
               PROJECTED_PTS=lambda d: (d._pts / d._gp) * (d._wgp / d._wsum)))

### 3.5 V2 board - value over replacement

Raw projected points ignore **scarcity**. A centre worth 1,500 is more valuable than a
guard worth 1,500 if good centres are rarer. VOR subtracts a per-position baseline
tied to league size and roster shape.

Let $d_p$ be the number of *dedicated* starting slots mapping to coarse position $p$
(PG, SG, G $\to$ G; SF, PF, F $\to$ F; C $\to$ C), and let $\mathcal{F}$ be the
remaining draftable slots (UTIL **and bench**), spread evenly over the three coarse
positions:

$$\sigma_p \;=\; d_p + \frac{|\mathcal{F}|}{3}
\qquad\Longrightarrow\qquad
k_p \;=\; \max\bigl(1,\; \operatorname{round}(T \cdot \sigma_p)\bigr)$$

$k_p$ is how many players at position $p$ the whole league drafts. The
**replacement-level** player is the one sitting at that rank, and VOR is the surplus
over him:

$$R_p \;=\; \hat{y}^{\,(p)}_{(k_p)}, \qquad\qquad
\boxed{\;\text{VOR}_i \;=\; \hat{y}_i \;-\; R_{p(i)}\;}$$

where $\hat{y}^{\,(p)}_{(k)}$ is the $k$-th largest projection among players at
position $p$ (clamped to the weakest if the pool is smaller). Players with no known
position are scored against an **overall** baseline
$k_0 = \max(1, \operatorname{round}(T \sum_p \sigma_p))$ - surfaced and counted,
never dropped.

> **Note the denominator.** VOR counts *all draftable* slots (including bench),
> because that is how many players actually leave the board. The lineup objective in
> 3.6 counts *only scoring* slots. Two different questions, two different counts.

In [ ]:
sigma_p            = rigid[p] + flexible / 3          # flexible includes UTIL + bench
replacement_rank   = max(1, round(NUM_TEAMS * sigma_p))
replacement_value  = nth_largest_projection_at(p, replacement_rank)   # clamped
VOR                = projected_points - replacement_value[position]
# Unknown-position players use an overall baseline and are counted in
# board.attrs['n_unknown_pos'] - surfaced, never silently dropped.

![Value by position](outputs/value_by_position.png)

### 3.6 V3 objective - what a roster is actually worth

In H2H points **only your starters score**. So a roster $R$ is worth its best *legal
starting lineup*, not the sum of all 13 players:

$$L(R) \;=\; \max_{\text{legal assignments}} \;\sum_{i \,\in\, \text{starters}} v_i$$

For this slot structure the maximum has a closed form. Dedicated slots can only be
filled by their own position, so they take that position's best; the flex (UTIL) slots
then take the best of everyone left over. With $d_p$ dedicated slots, $f$ flex slots
(**bench excluded - it does not score**), and $u$ the multiset of leftovers (players
not used in a dedicated slot, plus unknown-position players, who are flex-eligible
only):

$$\boxed{\;L(R) \;=\; \sum_{p \in \{G,F,C\}} \; \sum_{j=1}^{d_p} v^{(p)}_{(j)}
\;\;+\;\; \sum_{j=1}^{f} u_{(j)}\;}$$

**This objective is what makes V3 possible.** Under a sum-the-whole-roster objective,
$L(R) = \sum_{i \in R} v_i$ is *modular*, marginal value is constant, and greedy is
provably optimal - V3 would be pointless. The slot constraint makes $L$ **submodular**
(diminishing returns to depth), and that is exactly the structure a smarter engine can
exploit.

In [ ]:
def optimal_lineup(roster, value_by_id, position_by_id, league):
    dedicated, n_flex = starting_requirements(league)   # e.g. {G:3, F:3, C:1}, 3
    starters, leftovers = [], list(unknown_position_players)
    for position, capacity in dedicated.items():
        ranked = sorted(buckets[position], key=value, reverse=True)
        starters  += ranked[:capacity]                  # dedicated slots
        leftovers += ranked[capacity:]                  # surplus competes for flex
    starters += sorted(leftovers, key=value, reverse=True)[:n_flex]
    return starters, sum(value(p) for p in starters)

### 3.7 V3 engine - marginal value plus positional-run risk

At each of my picks, with current roster $R$ and available pool $A$ (in board order),
each candidate is scored on what he **adds to my lineup**, plus how fast his position
is drying up.

**Marginal lineup value** - the submodular gain:

$$\Delta_i \;=\; L\bigl(R \cup \{i\}\bigr) \;-\; L(R)$$

**Positional-run risk** - the cheap substitute for a full opponent simulation. Let $g$
be the number of opponent picks before my next turn (from the snake order), and assume
greedy opponents take the top $g$ players on the board. The number of position-$p$
players expected to disappear is

$$\tau_p \;=\; \bigl|\{\, j \in A_{1:g} \;:\; p(j) = p \,\}\bigr|$$

and the expected deterioration at that position between now and my next pick is

$$\text{run}_p \;=\; \max\Bigl(0,\;\; v^{(p)}_{(1)} \;-\; v^{(p)}_{(\tau_p + 1)}\Bigr)$$

(taking $v^{(p)}_{(\tau_p+1)} = 0$ if the pool runs out). Note $\text{run}_p$ depends
on the *position*, not the candidate - it ranks positions, while $\Delta_i$ ranks
players within one.

**The combined rule**, with $\lambda$ = `RUN_RISK_WEIGHT` $= 1.0$:

$$\boxed{\;\text{score}_i \;=\; \Delta_i \;+\; \lambda \cdot \text{run}_{p(i)} \cdot \mathbf{1}\!\left[\Delta_i > 0\right],
\qquad \text{pick } \arg\max_{i \in A} \text{score}_i\;}$$

**The indicator is not cosmetic.** Without it the engine grabbed a surplus guard it
could never start, purely because guards were running out - a real bug caught by the
tests. Run-risk must never resurrect a pick with zero marginal value.

In [ ]:
for cand in available:
    marginal = lineup_value(my_roster + [cand]) - base

    # Gate: run-risk applies only to candidates who actually improve the lineup.
    if marginal > 0:
        run = positional_run_risk(pos[cand], available, gap_to_next_pick)
        score = marginal + RUN_RISK_WEIGHT * run
    else:
        score = marginal

**Snake draft order.** With $T$ teams and $\text{RS}$ roster slots, the order reverses
each round, so team $t$'s overall pick number in round $r$ is

$$P(t, r) \;=\; (r-1)\,T \;+\;
\begin{cases}
t & r \text{ odd} \\[2pt]
T - t + 1 & r \text{ even}
\end{cases}
\qquad r = 1, \dots, \text{RS}$$

and the gap between my consecutive picks alternates between short and long - which is
precisely what $g$ measures and why run-risk matters more at the turn.

### 3.8 V4 model - aging curve and empirical-Bayes shrinkage

V4 refines the **rate** $\hat{r}_i$ from V2, leaving $\hat{G}_i$ untouched:

$$\hat{y}^{\,V4}_i(N) \;=\; \underbrace{A_i}_{\text{aging}} \cdot
\underbrace{\tilde{r}_i}_{\text{shrunk rate}} \cdot \; \hat{G}_i$$

#### (a) The aging curve

For every consecutive-season pair a player *completed* (both seasons before $N$, both
with $G \ge 20$, and $r_{i,s} > 0$), take the year-over-year rate ratio:

$$\rho_{i,s} \;=\; \frac{r_{i,s+1}}{r_{i,s}}$$

Summarise by the player's age in the first season of the pair, using the **median** for
robustness to outliers, then smooth over a $\pm h$ window ($h = 1$ year):

$$\tilde{m}(a) \;=\; \operatorname{median}\bigl\{\, \rho_{i,s} \;:\; \operatorname{round}(\text{age}_{i,s}) = a \,\bigr\}$$

$$m(a) \;=\; \frac{1}{|H_a|}\sum_{a' \in H_a} \tilde{m}(a'), \qquad
H_a = \bigl\{\, a' \in [a-h,\, a+h] \;:\; \tilde{m}(a') \text{ exists} \,\bigr\}$$

with $m(a) = 1$ (no adjustment) where there is no data, and $a$ clamped to
$[19, 38]$. Applying it means **chaining** the multipliers from the player's most
recent observed age $\alpha_i$ forward $\Delta_i = N_{\text{year}} - s^{\text{last}}_{\text{year}}$ years:

$$\boxed{\;A_i \;=\; \prod_{a \,=\, \alpha_i}^{\alpha_i + \Delta_i - 1} m\bigl(\operatorname{clamp}(a)\bigr)\;}$$

(so a player who missed a whole season is aged forward two years, not one).

In [ ]:
def build_aging_curve(totals, bios, target_season, league, min_games=20):
    ppg = _ppg_by_player_season(totals).loc[lambda d: d[COL_SEASON] < target_season]
    pairs = (ppg.assign(_next=lambda d: d[COL_SEASON].map(next_season))
                .merge(ppg, left_on=[COL_PLAYER_ID, "_next"],
                       right_on=[COL_PLAYER_ID, COL_SEASON], suffixes=("_1", "_2")))
    pairs = pairs.assign(_ratio=lambda d: d["PPG_2"] / d["PPG_1"])
    # KNOWN FLAW - survivorship bias: only players who appeared in BOTH seasons are
    # here, so players who age out are missing and late decline is understated.
    return smooth(pairs.groupby("_age")["_ratio"].median())

![Aging curve](outputs/aging_curve.png)

The curve behaves exactly as theory predicts - improvement into the early 20s, a peak
near 26 ($m(26) \approx 0.995$), decline through the 30s ($m(30) \approx 0.89$,
$m(34) \approx 0.84$) - **and it visibly exposes its own bias**: the multiplier *rises
again* at 36-38 because only the strongest old players survive to be measured. That
artefact is documented rather than smoothed away.

#### (b) Empirical-Bayes shrinkage

Regress each player's rate toward a **role prior** (his position's pooled rate), with
weight determined by sample size in games. With $n_i = \sum_{s \in \mathcal{S}_i} G_{i,s}$
and pseudo-count $k = 25$ games:

$$\mu_p \;=\; \frac{\sum_{i \,:\, p(i)=p} \sum_{s} T_{i,s}}{\sum_{i \,:\, p(i)=p} \sum_{s} G_{i,s}}
\qquad\qquad
\boxed{\;\tilde{r}_i \;=\; \frac{n_i\,\hat{r}_i \;+\; k\,\mu_{p(i)}}{n_i + k}\;}$$

This is the standard conjugate shrinkage form: the weight on the player's own data is
$n_i/(n_i+k)$, so $\tilde{r}_i \to \hat{r}_i$ as $n_i \to \infty$ and
$\tilde{r}_i \to \mu_p$ for tiny samples. $k$ is interpretable as "how many games of
prior evidence" - here a third of a season.

**It did not work, and the formula shows why.** The prior $\mu_p$ is a *positional
mean*, so an elite player is pulled toward average even with 70 games
($n/(n+k) = 0.74$, i.e. 26% of the way). A player's own multi-year record is a far
better prior than "the average guard" - and $\hat{r}_i$ already *is* that. Shrinkage
was measured, found harmful, and switched off (`V4_USE_SHRINKAGE = False`).

In [ ]:
# Shrink first (clean the rate estimate), then age (project it forward).
if use_shrinkage:
    prior = pos_prior.get(position, overall_prior)
    ppg = (n * ppg + k * prior) / (n + k)
if use_aging:
    ppg *= _aging_factor(last_age[pid], years_forward, curve, league)
projected_points = ppg * projected_games

### 3.9 V5 - the multi-season distribution

A single season is high variance, so the trustworthy verdict is a **distribution**
across held-out seasons $\mathcal{N}$, against named benchmarks
$b \in \{\text{prior totals},\, \text{ADP},\, \text{random}\}$ - where
**prior totals is the baseline** and random is only a floor. All drafts are scored
with the same lineup objective $L(\cdot)$ on **actual** points:

$$\text{margin}_b(N) \;=\; L^{\text{smart}}(N) \;-\; L^{b}(N)
\qquad\qquad
\text{pct}_b(N) \;=\; 100\left(\frac{L^{\text{smart}}(N)}{L^{b}(N)} - 1\right)$$

$$\overline{\text{margin}}_b = \frac{1}{|\mathcal{N}|}\sum_{N \in \mathcal{N}} \text{margin}_b(N),
\qquad
\text{s.d.}_b = \sqrt{\frac{1}{|\mathcal{N}|}\sum_{N \in \mathcal{N}}\bigl(\text{margin}_b(N) - \overline{\text{margin}}_b\bigr)^2}$$

(population standard deviation, `ddof=0`, since these are all the seasons tested, not
a sample from them.)

The **baseline** $b = \text{prior totals}$ is the naive model: a greedy snake draft off
$\hat{y}_i(N) = y_i(N-1)$, i.e. "take whoever scored most last season". It is the bar a
competent human sets, so $\text{margin}_{\text{prior}}$ is the headline number.

The random benchmark is retained only as a **sanity floor**, and is itself an average
over $M = 200$ seeded draft simulations so that no single random draft's noise shows
through:

$$L^{\text{random}}(N) \;=\; \frac{1}{M} \sum_{m=1}^{M} L\bigl(R^{\text{rand},\,m}(N)\bigr)$$

Clearing it shows the pipeline is wired up, nothing more - nobody drafts randomly.

### 3.10 Evaluation metrics

**Projection accuracy is not one number.** Scored on the common player set
$\mathcal{P}$ - players every method can project *and* who have an actual total - so
comparisons are apples-to-apples. Three families are reported, because MAE alone is a
weak test here:

$$\text{MAE} = \frac{1}{|\mathcal{P}|}\sum_{i} \bigl|\hat{y}_i - y_i\bigr|
\qquad
\text{RMSE} = \sqrt{\frac{1}{|\mathcal{P}|}\sum_{i} \bigl(\hat{y}_i - y_i\bigr)^2}
\qquad
\text{bias} = \frac{1}{|\mathcal{P}|}\sum_{i} \bigl(\hat{y}_i - y_i\bigr)$$

**1. Error size.** MAE is the typical error, RMSE penalises catastrophic misses
quadratically, the **median** absolute error is robust to the injury blow-ups that
fatten the tail, and the 90th percentile *is* that tail. **Bias is signed**, so a
projection that is systematically optimistic shows up - something no absolute-error
metric can reveal.

**2. Association.** Pearson, $R^2$, and **Spearman rank correlation**. Spearman is
the closest single number to "is this a good draft board".

**3. Top-of-board.** The decisive family, because a board is a *ranking* and only
$\text{NUM\_TEAMS} \times \text{ROSTER\_SIZE} = 156$ players are ever drafted:

$$\text{precision@}k = \frac{|\text{top}_k(\hat{y}) \cap \text{top}_k(y)|}{k}
\qquad
\text{NDCG@}k = \frac{\sum_{i=1}^{k} y_{\pi(i)} / \log_2(i+1)}
                      {\sum_{i=1}^{k} y_{(i)} / \log_2(i+1)}$$

where $\pi$ is the projected ordering and $y_{(i)}$ the actual totals sorted
descending. NDCG discounts by rank, so getting the first pick right counts far more
than the hundredth.

**Why this matters:** a projection that doubles every total and adds 500 has a
catastrophic MAE and produces an **identical board**. Optimising MAE is optimising
the wrong thing.

**Decision quality.** Accuracy is still only a *proxy*; what matters is the lineup
points the resulting draft produces, which is what the head-to-head measures.

### 3.11 Backtest design

- **Time-ordered** always: train on earlier seasons, score on a later one. Never a
  random split across time.
- **Full-draft simulation**: every team drafts (opponents greedily), rather than
  replaying a historical draft - no faithful replay exists, so the choice is stated
  explicitly. **The engine is only as valid as that opponent model**, which is the
  binding constraint on all V3 results.
- **The baseline is a naive model, not a random draft**: the prior-season-totals
  draft. Nothing is reported in isolation; every result is a margin over it. ADP is a
  secondary benchmark, and the random draft is kept only as a sanity floor beneath
  both - beating it is a wiring check, not evidence.
- **Distribution, not a point estimate**: three held-out seasons, reported with spread
  and win counts.

## 4. Results

### 4.1 Projection accuracy (time-ordered, target 2023-24, n=450)

In [ ]:
import pandas as pd

# Pooled over 13 target seasons. k = 156 drafted, round 1 = 12.
accuracy = pd.DataFrame({
    "method":   ["V1 prior total", "V2 PPG x GP", "V4 aging",
                 "V4 shrinkage", "V4 aging + shrink"],
    "MAE":      [508.9, 521.1, 501.8, 522.1, 504.9],
    "MedAE":    [384.3, 410.8, 401.2, 423.8, 408.6],
    "bias":     [ 34.9,  13.3,   9.1,  29.4,  28.5],
    "Spearman": [0.731, 0.712, 0.731, 0.711, 0.730],
    "P@156":    [0.735, 0.714, 0.731, 0.713, 0.726],
    "NDCG@156": [0.884, 0.878, 0.888, 0.877, 0.886],
    "P@round1": [0.532, 0.558, 0.577, 0.545, 0.577],
})
accuracy

![Projection accuracy](outputs/projection_accuracy.png)

**No variant dominates, and the metrics disagree - which is itself the finding.**
V4's aging curve wins MAE, RMSE, the tail, the bias and the rank-weighted top of the
board (NDCG, first-round precision). V1 wins the median error and the ordering
*within* the drafted pool. Overall Spearman is a dead tie at 0.731.

That is coherent rather than contradictory: aging mainly fixes the *large* misses -
ageing players whose prior-season total badly overstates them - so it wins the
mean-based and tail metrics while slightly losing the median. Shrinkage loses on
every family and stays switched off.

**Two things only the new metrics could show.**

*V1 is systematically optimistic*: bias **+34.9** points per player against V4
aging's **+9.1**. Prior-season totals over-predict, because last season's top
finishers were disproportionately both good *and* healthy, and nobody is getting
younger. No absolute-error metric can reveal a direction.

*The board is far weaker at the top than it looks*: overall Spearman is ~0.73, but
ordering the players you actually draft is only ~0.55-0.58, and
**`precision@round1` is ~0.53-0.58** - barely half of your projected first-rounders
finish in the real top twelve. That belongs in the honest-framing conversation more
than any MAE does.

### 4.2 Draft results, 2023-24 (whole-roster actual points)

In [ ]:
# The baseline is the naive model: the V1 greedy draft IS the prior-season-totals
# draft, so it is the reference rather than a competitor. Random is a floor beneath.
NAIVE_BASELINE = 31991.0

drafts = pd.DataFrame({
    "draft": ["Naive baseline (prior totals)", "V2 greedy (VOR)",
              "Random floor (mean of 200)"],
    "actual_points": [NAIVE_BASELINE, 31346.2, 13888.9],
})
drafts["vs_naive_%"] = (
    (drafts.actual_points / NAIVE_BASELINE - 1) * 100).round(1)
drafts

### 4.3 The trustworthy read: multi-season distribution

Scored on the **starting-lineup** objective (only starters count), across three
held-out seasons.

In [ ]:
# 13 target seasons (2013-14 .. 2025-26) on history from 2010-11.
multi = pd.DataFrame({
    "benchmark":   ["naive baseline (prior totals)", "ADP proxy", "random floor"],
    "mean_margin": [-74.2, 477.9, 15085.9],
    "mean_pct":    [-0.2,   2.1,    153.0],
    "seasons_won": ["8/13", "7/13", "13/13"],
})
multi

![Backtest distribution](outputs/backtest_distribution.png)

**The same-board result did not survive more data.** On three target seasons this
benchmark showed the engine beating the naive baseline by +4.8%. On thirteen it is
**-0.2% (p = 0.96, 95% CI [-7.8%, +7.5%])**, with per-season swings from -20.3% to
+19.9%. The apparent edge was noise. So was the V3-beats-V2-greedy result, which
went from 3/3 seasons to 7/13.

That is a limitation of the *experiment*, not a verdict on the engine: every team
drafts off the same board here, so changing the board changes the whole league and
my slot's advantage largely cancels.

### 4.4 The head-to-head - holding the league fixed

`head_to_head_backtest` fixes it: my eleven opponents all draft naively, and only my
board and strategy change - **averaged over all twelve draft slots**, because within
one season the slot-to-slot spread is ~12 percentage points, as large as the
variation between seasons. That is **12 x 13 = 156 drafts per variant**.

| My version | Mean % vs naive | 95% CI | p | Seasons won | Slot-seasons positive |
|---|---|---|---|---|---|
| V2 VOR board, greedy | **-11.8%** | [-21.9, -1.7] | 0.026 | 2/13 | 28% |
| V4 aging board, greedy | **-1.4%** | [-11.2, +8.3] | 0.755 | 6/13 | 48% |
| V3 engine on V2 board | **+7.8%** | [+4.3, +11.3] | 0.0004 | 12/13 | 72% |
| **V4 board + V3 engine** | **+14.6%** | **[+9.0, +20.3]** | **0.0001** | **13/13** | **85%** |

**The engine is what buys the edge, and it wins every season tested.** Boards
drafted greedily are worth nothing or worse. The better projection compounds through
the engine - V4's board nearly doubles the margin - but on its own it is neutral.

**Who sets the starting lineup?** Scoring picks the best legal lineup and sums actual
points, but "best" needs a sort key. By default it is the *actual* totals - the best
lineup the roster could have produced, an upper bound. The alternative is the board's
own projection: the lineup you would have set in October, a lower bound. Both are
computed (`lineup_value_source`), and the margin barely moves:

| Head-to-head edge | Hindsight | Projection-picked |
|---|---|---|
| V3 engine on V2 board | +7.8% | +6.7% |
| **V4 board + V3 engine** | **+14.6%** | **+15.0%** |

Hindsight inflates every roster's absolute points, but it inflates the naive baseline
at least as much as the engine, so the conclusion does not depend on the assumption.
Note this is a *scoring* choice: no draft decision anywhere sees target-season data.

### 4.5 Summary scorecard

| Question | Answer |
|---|---|
| Does it beat naive, league held fixed? | **Yes, +14.6%**, 13/13 seasons, p = 0.0001 |
| Does it beat naive when everyone upgrades? | **No, -0.2%**, p = 0.96 - that experiment measures nothing |
| Does it clear the random floor? | Yes - +153%, 13/13, but that is a wiring check |
| Did a sophisticated projection help accuracy? | **Yes** - V4 aging 501.8 vs V1 508.9 pooled |
| Did that convert into a better draft? | **Only via the engine** - +14.6% with it, +7.8% without |
| Is the VOR board a draft order? | **No** - greedy off VOR is **-11.8%**, p = 0.026 |

## 5. Discussion

### What worked, and why

**The decision engine paid off; the fancier projections did not.** V3's marginal-value
engine beat greedy in every season tested, because greedy optimises the wrong thing -
it accumulates talent it cannot start. Roster *construction* turned out to be a bigger
lever than forecasting. Against a league held fixed, it is the **only** component that
beats drafting naively.

**Value over replacement earned its place - inside the engine, not as a ranking.**
Positional scarcity is real and cheap to model, but the head-to-head showed that
*drafting straight off* the VOR board is 11.8% worse than naive (p = 0.026). The replacement levels
tell you why: G 1,808 / F 1,552 / C 826 on 2023-24, so VOR inflates centres and you
open Jokic, Embiid, Vucevic while Doncic and Gilgeous-Alexander sit there. Against a
market pricing players by last season's totals, that is a systematic reach. The engine
consumes scarcity correctly because it asks what a candidate adds to *my lineup*; the
board alone does not.

### What did not work, and why that is still a result

**Prior-season total is a brutally strong baseline - and was undefeated until it
was not.** Recency weighting, the rate x durability split, and shrinkage all failed to
beat it on MAE over the original three seasons. On thirteen, V4's aging curve does beat
it (501.8 vs 508.9 pooled), and that better forecast now also produces a better draft -
but only once the engine consumes it. The original failures remain instructive:

- **Multi-season blending lags genuine change.** NBA players break out and change roles;
  averaging three seasons dilutes exactly the recent signal that matters.
- **Shrinkage toward a positional mean is the wrong prior.** It drags genuinely elite
  players toward average. A player's own multi-year record is a far better prior than
  "the average guard" - and V2 already was that.
- **The games-played estimate adds variance** of its own, sometimes more than the
  durability signal it recovers.

This is the "keep only what helps" rule doing its job: three components were built,
measured, and two were switched off. Reporting that honestly is more valuable than
shipping complexity that quietly loses.

### Why the baseline is a naive model, not a random draft

+172% vs random is a **low bar** and would be misleading as a headline. Nobody drafts
randomly. Against the naive model - someone who simply sorts by last season's totals -
the edge is ~3%, and it *loses* one season in three. Both numbers are true; only the
second one is decision-relevant, which is why the naive draft is now **the baseline**
and the random draft is demoted to a sanity floor reported beneath it.

### Known limitations

| Limitation | Impact |
|---|---|
| **Survivorship bias in the aging curve** | Late-career decline understated; visible as the 36-38 uptick |
| **Opponent model is greedy bots** | The engine's edge would shrink against sharper opponents |
| **Coarse G/F/C positions** | True Yahoo multi-position eligibility would sharpen VOR and lineups |
| **Bench value ignored** | Streaming and injury cover are real H2H levers, unmodelled |
| **No rookie signal** | Players without NBA history are absent from the board |
| **Three seasons only** | Every estimate would tighten with more history |
| **ADP is a proxy** | Real draft-position data needs a live source |
| **Yahoo path unexecuted** | Scaffolded and unit-tested, but needs the one-time OAuth |

### On availability

Games played is the largest and least predictable source of variance in the whole
system. V2 modelled it explicitly and *still* did not beat a stale total - which says
less about the method than about how hard availability is to forecast a season ahead.

## 6. Conclusion

A complete, leakage-free pipeline was built and validated end to end: multi-source
ingestion with a verified schema and a Parquet cache, a scoring layer, four
progressively more sophisticated projections, a constrained draft engine, and a
multi-season backtest against named benchmarks - 58 tests, all passing.

**The four findings that matter:**

1. **Roster construction beat forecasting.** The smarter *draft engine* delivered a
   consistent win; the smarter *projections* did not. Effort spent on the decision
   layer outperformed effort spent on the prediction layer.
2. **A better board is not a better draft order.** Drafting greedily off the value-
   over-replacement board is 11.8% *worse* than naive against a league that prices
   players by last season's totals (p = 0.026). Scarcity belongs inside the engine,
   not in the ranking - a distinction the original same-board backtest could not see.
3. **The naive baseline held on accuracy for three seasons, then fell.** Prior-season
   total survived every challenger on the original window; two more seasons of data
   and V4's aging curve beats it. The negative result was real, and it was temporary -
   which is why the evaluation is re-run rather than quoted.
4. **The edge is real, and measuring it needed care.** **+14.6%** over a competent
   baseline with the league held fixed, winning all 13 seasons, positive in 85% of
   156 slot-seasons, p = 0.0001. Getting a trustworthy number took three separate
   fixes to the *measurement* rather than the model: benchmark against a naive
   baseline instead of a random draft; hold the opposing league fixed instead of
   upgrading everyone at once; and average all twelve draft slots, because
   slot-to-slot variance within a season rivals the variance between seasons. Each
   fix changed the answer materially, and the same-board numbers this project once
   reported with confidence turned out to be noise. A fourth check - scoring the
   lineup from the projection instead of from hindsight - turned out *not* to matter
   (+15.0% against +14.6%), which is worth knowing too.

**What would move the needle next**, in order of expected value: true Yahoo
multi-position eligibility (sharpens both VOR and the lineup objective); a real ADP
source for a fair market benchmark; more seasons of history; and modelling bench
value, since in-season management is where H2H leagues are actually won.

The project does what it set out to do - and it is honest about how much that is
worth.